## Bookings Forecast Assumptions

In [74]:
import pandas as pd
import numpy as np
import openpyxl

In [75]:
PRE_Q = {
    "2026-Q1": {"push_rate": 0.0, "dist": {3: 0.60, 6: 0.40, 9: 0.00}},
    "2026-Q2": {"push_rate": 0.46, "dist": {3: 0.60, 6: 0.40, 9: 0.00}},
    "2026-Q3": {"push_rate": 0.46, "dist": {3: 0.60, 6: 0.40, 9: 0.00}},
    "2026-Q4": {"push_rate": 0.46, "dist": {3: 0.60, 6: 0.40, 9: 0.00}},
}

IN_Q = {
    "2026-Q1": {"push_rate": 0.50, "dist": {3: 0.60, 6: 0.40, 9: 0.00}}, 
    "2026-Q2": {"push_rate": 0.55, "dist": {3: 0.53, 6: 0.40, 9: 0.00}},
    "2026-Q3": {"push_rate": 0.55, "dist": {3: 0.60, 6: 0.40, 9: 0.00}},
    "2026-Q4": {"push_rate": 0.60, "dist": {3: 0.60, 6: 0.40, 9: 0.00}},
}


## Win Rate Assumptions

In [76]:
Pre_Q_Win_Rates = {
    # ── Americas ─────────────────────────────────────────────
    "AMS Core East Canada": {
        "FY25/Q1": 0.08, "FY25/Q2": 0.08, "FY25/Q3": 0.13, "FY25/Q4": 0.22,
        "FY26/Q1": 0.08, "FY26/Q2": 0.08, "FY26/Q3": 0.13, "FY26/Q4": 0.22
    },
    "AMS Core East LATAM": {
        "FY25/Q1": 0.07, "FY25/Q2": 0.11, "FY25/Q3": 0.17, "FY25/Q4": 0.07,
        "FY26/Q1": 0.07, "FY26/Q2": 0.11, "FY26/Q3": 0.17, "FY26/Q4": 0.07
    },
    "AMS Core East Northeast": {
        "FY25/Q1": 0.02, "FY25/Q2": 0.11, "FY25/Q3": 0.10, "FY25/Q4": 0.12,
        "FY26/Q1": 0.02, "FY26/Q2": 0.11, "FY26/Q3": 0.10, "FY26/Q4": 0.12
    },
    "AMS Core South Southeast": {
        "FY25/Q1": 0.05, "FY25/Q2": 0.10, "FY25/Q3": 0.14, "FY25/Q4": 0.17,
        "FY26/Q1": 0.05, "FY26/Q2": 0.10, "FY26/Q3": 0.14, "FY26/Q4": 0.17
    },
    "AMS Core South TOLA": {
        "FY25/Q1": 0.09, "FY25/Q2": 0.11, "FY25/Q3": 0.09, "FY25/Q4": 0.08,
        "FY26/Q1": 0.09, "FY26/Q2": 0.11, "FY26/Q3": 0.09, "FY26/Q4": 0.08
    },
    "AMS Core West Central": {
        "FY25/Q1": 0.11, "FY25/Q2": 0.06, "FY25/Q3": 0.03, "FY25/Q4": 0.28,
        "FY26/Q1": 0.11, "FY26/Q2": 0.06, "FY26/Q3": 0.03, "FY26/Q4": 0.28
    },
    "AMS Core West Pacific": {
        "FY25/Q1": 0.05, "FY25/Q2": 0.09, "FY25/Q3": 0.07, "FY25/Q4": 0.11,
        "FY26/Q1": 0.05, "FY26/Q2": 0.09, "FY26/Q3": 0.07, "FY26/Q4": 0.11
    },
    "AMS DevOps": {
        "FY25/Q1": 0.55, "FY25/Q2": 0.00, "FY25/Q3": 0.28, "FY25/Q4": 0.00,
        "FY26/Q1": 0.55, "FY26/Q2": 0.00, "FY26/Q3": 0.28, "FY26/Q4": 0.00
    },
    "AMS Public Sector - SLED": {
        "FY25/Q1": 0.47, "FY25/Q2": 0.47, "FY25/Q3": 0.47, "FY25/Q4": 0.50,
        "FY26/Q1": 0.47, "FY26/Q2": 0.47, "FY26/Q3": 0.47, "FY26/Q4": 0.50
    },
    "AMS Public Sector - FED": {
        "FY25/Q1": 0.50, "FY25/Q2": 0.50, "FY25/Q3": 0.55, "FY25/Q4": 0.55,
        "FY26/Q1": 0.50, "FY26/Q2": 0.50, "FY26/Q3": 0.55, "FY26/Q4": 0.55
    },
    "AMS SeaLights": {
        "FY25/Q1": 0.10, "FY25/Q2": 0.10, "FY25/Q3": 0.10, "FY25/Q4": 0.04,
        "FY26/Q1": 0.10, "FY26/Q2": 0.10, "FY26/Q3": 0.10, "FY26/Q4": 0.04
    },
    
    # ── APAC ─────────────────────────────────────────────────
    "APAC ANZ": {
        "FY25/Q1": 0.05, "FY25/Q2": 0.14, "FY25/Q3": 0.14, "FY25/Q4": 0.20,
        "FY26/Q1": 0.05, "FY26/Q2": 0.14, "FY26/Q3": 0.14, "FY26/Q4": 0.20
    },
    "APAC Asia": {
        "FY25/Q1": 0.11, "FY25/Q2": 0.11, "FY25/Q3": 0.16, "FY25/Q4": 0.13,
        "FY26/Q1": 0.11, "FY26/Q2": 0.11, "FY26/Q3": 0.16, "FY26/Q4": 0.13
    },
    "APAC DevOps": {
        "FY25/Q1": 0.10, "FY25/Q2": 0.10, "FY25/Q3": 0.10, "FY25/Q4": 0.10,
        "FY26/Q1": 0.10, "FY26/Q2": 0.10, "FY26/Q3": 0.10, "FY26/Q4": 0.10
    },
    "APAC Japan": {
        "FY25/Q1": 0.06, "FY25/Q2": 0.03, "FY25/Q3": 0.24, "FY25/Q4": 0.24,
        "FY26/Q1": 0.06, "FY26/Q2": 0.03, "FY26/Q3": 0.24, "FY26/Q4": 0.24
    },
    
    # ── EMEA ─────────────────────────────────────────────────
    "EMEA Core Alps CEE": {
        "FY25/Q1": 0.03, "FY25/Q2": 0.08, "FY25/Q3": 0.12, "FY25/Q4": 0.05,
        "FY26/Q1": 0.03, "FY26/Q2": 0.08, "FY26/Q3": 0.12, "FY26/Q4": 0.05
    },
    "EMEA Core BeNeLux Nordics": {
        "FY25/Q1": 0.04, "FY25/Q2": 0.16, "FY25/Q3": 0.14, "FY25/Q4": 0.12,
        "FY26/Q1": 0.04, "FY26/Q2": 0.16, "FY26/Q3": 0.14, "FY26/Q4": 0.12
    },
    "EMEA Core France Emerging": {
        "FY25/Q1": 0.11, "FY25/Q2": 0.36, "FY25/Q3": 0.17, "FY25/Q4": 0.16,
        "FY26/Q1": 0.11, "FY26/Q2": 0.36, "FY26/Q3": 0.17, "FY26/Q4": 0.16
    },
    "EMEA Core Germany": {
        "FY25/Q1": 0.08, "FY25/Q2": 0.12, "FY25/Q3": 0.14, "FY25/Q4": 0.15,
        "FY26/Q1": 0.08, "FY26/Q2": 0.12, "FY26/Q3": 0.14, "FY26/Q4": 0.15
    },
    "EMEA Core Middle East": {
        "FY25/Q1": 0.05, "FY25/Q2": 0.33, "FY25/Q3": 0.15, "FY25/Q4": 0.27,
        "FY26/Q1": 0.05, "FY26/Q2": 0.33, "FY26/Q3": 0.15, "FY26/Q4": 0.27
    },
    "EMEA Core UKI": {
        "FY25/Q1": 0.20, "FY25/Q2": 0.08, "FY25/Q3": 0.06, "FY25/Q4": 0.17,
        "FY26/Q1": 0.20, "FY26/Q2": 0.08, "FY26/Q3": 0.06, "FY26/Q4": 0.17
    },
    "EMEA DevOps": {
        "FY25/Q1": 0.00, "FY25/Q2": 0.12, "FY25/Q3": 0.04, "FY25/Q4": 0.11,
        "FY26/Q1": 0.00, "FY26/Q2": 0.12, "FY26/Q3": 0.04, "FY26/Q4": 0.11
    }
}

In [77]:
RULES = []

In [78]:
# Import libraries
import logging
from pathlib import Path
import pandas as pd
import numpy as np
from pandas.tseries.offsets import DateOffset

def find_root(markers=("Input files","output files",".project-root")):
    p = Path.cwd().resolve()
    for q in (p, *p.parents):
        if any((q/m).exists() for m in markers): return q
    raise RuntimeError("Project root not found")

ROOT        = find_root()
INPUT_FILE  = ROOT / "output files" / "Data Ingestion 2025 Weighted.xlsx"
OUTPUT_FILE = ROOT / "Open Pipeline Forecast Product V10.xlsx"

# Column names from our data
STAGE_COL = "Stage"              # Which stage is the deal in?
DATE_COL = "Opp Close Date"      # When should it close?
VALUE_COL = "Product NACV"       # How much is it worth?

# Dimensions that describe each opportunity (removed Region - not used anymore)
ROW_KEYS = ["Territory", "Tier", "Product", "Source", "Deal Type"]
ROW_DIMS = ROW_KEYS + [STAGE_COL]

today = pd.Timestamp.today().normalize()

months_back = 1  # 0=this month, 1=last month, etc.
start_month = today.replace(day=1) - pd.DateOffset(months=months_back)

forecast_end = pd.Timestamp(2026, 12, 31)

# Create month grid starting from start_month (last month if months_back=1)
month_grid = pd.date_range(start_month, forecast_end, freq="MS")

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s"
)

print(f"✓ Setup complete!")
print(f"✓ Today: {today.strftime('%B %d, %Y')}")
print(f"✓ Forecast period: {start_month.strftime('%b %Y')} to {forecast_end.strftime('%b %Y')}")
print(f"✓ Total months: {len(month_grid)}")
print(f"✓ Note: All months BEFORE {start_month.strftime('%b %Y')} will be dropped")


✓ Setup complete!
✓ Today: February 26, 2026
✓ Forecast period: Jan 2026 to Dec 2026
✓ Total months: 12
✓ Note: All months BEFORE Jan 2026 will be dropped


## 🧹 Function: Clean Column Names

**What we're doing:**
Excel files sometimes have extra spaces in column names. This fixes them.

**Example:** `"Product  NACV"` → `"Product NACV"`

In [79]:
def clean_headers(df):
    """Remove extra spaces and clean up column names from Excel"""
    df = df.rename(columns=lambda c: " ".join(str(c).replace("\u00a0", " ").split()))
    df = df.rename(columns=str.strip)
    return df

# Quick test
test_df = pd.DataFrame({"  Name ": [1], "Product  Value": [2]})
print("Before:", list(test_df.columns))
print("After: ", list(clean_headers(test_df).columns))

Before: ['  Name ', 'Product  Value']
After:  ['Name', 'Product Value']


## 📊 Function: Align Data to Month Grid

**What we're doing:**
Ensure all dataframes have the same month columns (only future months).
Missing months get filled with 0.

**Why:** Different calculations might produce different month ranges.
We need them consistent for combining.

In [80]:
def align_to_months(df):
    """
    Align dataframe to our forecast month grid.
    Only includes current month forward - all past months are excluded.
    """
    return df.reindex(columns=month_grid, fill_value=0)

# Example
sample = pd.DataFrame({
    pd.Timestamp("2025-10-01"): [100],
    pd.Timestamp("2025-12-01"): [300],
})

print(f"Before: {len(sample.columns)} months")
aligned = align_to_months(sample)
print(f"After: {len(aligned.columns)} months (includes all months in forecast range)")

Before: 2 months
After: 12 months (includes all months in forecast range)


## 📉 Functions: Push Rates and Distribution

**What we're doing:**
When deals don't close on time, they "push" to future months.

**Why this matters:** Models real-world deal slippage behavior.

In [81]:
def get_push_rate(columns, push_rules):
    """
    For each month, look up what % of deals typically push out.
    Returns a Series mapping month → push_rate
    """
    rates = {}
    for col in columns:
        quarter_label = str(col.to_period("Q")).replace("Q", "-Q")
        quarter_rules = push_rules.get(quarter_label, {})
        rates[col] = quarter_rules.get("push_rate", 0)
    return pd.Series(rates)


def get_push_distribution(columns, push_rules, lag_months):
    """
    For each month, get what % of pushed deals land at a specific lag.
    
    Args:
        lag_months: 3, 6, or 9 (how many months in the future)
    
    Example: If 20% of deals push and lag=3, then they land 3 months later
    """
    distribution = {}
    for col in columns:
        quarter_label = str(col.to_period("Q")).replace("Q", "-Q")
        quarter_rules = push_rules.get(quarter_label, {})
        dist_rules = quarter_rules.get("dist", {})
        distribution[col] = dist_rules.get(lag_months, 0)
    return pd.Series(distribution)

print("✓ Push rate functions defined")

✓ Push rate functions defined


## 💼 Function: Extract Open Pipeline

**What we're doing:**
Extract all currently open opportunities, organized by expected close month.

**Output structure:**
- Rows: Territory/Tier/Product/Source/Deal Type/Stage combinations
- Columns: Months (Oct-2025, Nov-2025, Dec-2025, ...)
- Values: Total pipeline value

**Key:** Only includes months from current month forward (past months dropped).

In [82]:
def get_existing_pipeline(df):
    """
    Get all currently open opportunities, organized by close month.
    Only includes current month and future months.
    """
    # Filter to Open stage only
    open_opps = df[df[STAGE_COL] == "Open"].copy()
    
    # Convert dates
    open_opps[DATE_COL] = pd.to_datetime(open_opps[DATE_COL], errors="coerce")
    
    # Pivot to create month-by-month view
    pivot = open_opps.pivot_table(
        index=ROW_DIMS,
        columns=DATE_COL,
        values=VALUE_COL,
        aggfunc="sum",
        fill_value=0
    ).sort_index(axis=1).sort_index()
    
    # Align to our forecast months (drops all past months automatically)
    return align_to_months(pivot)

print("✓ get_existing_pipeline() function defined")
print("  Extracts open pipeline for forecast period only")

✓ get_existing_pipeline() function defined
  Extracts open pipeline for forecast period only


## ✅ Function: Extract Closed-Won Pipeline

**What we're doing:**
Extract all deals that already closed as "Closed Won". These are ACTUALS.

**Why this matters:**
- For past months, we want actual results (not forecasts)
- Allows us to compare forecast accuracy
- Creates a clean combined view in the final bookings sheet

In [83]:
def get_closed_won_pipeline(df):
    """
    Get all closed-won deals by their actual close month.
    These are ACTUALS - what really happened.
    """
    # Filter to Closed Won stage only
    closed_won = df[df[STAGE_COL] == "Closed Won"].copy()
    
    # Convert dates
    closed_won[DATE_COL] = pd.to_datetime(closed_won[DATE_COL], errors="coerce")
    
    # Pivot to create month-by-month view
    pivot = closed_won.pivot_table(
        index=ROW_DIMS,
        columns=DATE_COL,
        values=VALUE_COL,
        aggfunc="sum",
        fill_value=0
    ).sort_index(axis=1).sort_index()
    
    # Align to our full month grid
    return align_to_months(pivot)

print("✓ get_closed_won_pipeline() function defined")
print("  Extracts actual closed-won deals for all months")

✓ get_closed_won_pipeline() function defined
  Extracts actual closed-won deals for all months


## ✅ Function: Extract Closed-Lost Pipeline

**What we're doing:**
Extract all deals that already closed as "Closed Lost". These are ACTUALS.

**Why this matters:**
- For past months, we want actual results (not forecasts)
- Allows us to compare forecast accuracy
- Creates a clean combined view in the final bookings sheet

In [84]:
def get_closed_lost_pipeline(df):
    """
    Get all closed-won deals by their actual close month.
    These are ACTUALS - what really happened.
    """
    # Filter to Closed Won stage only
    closed_lost = df[df[STAGE_COL] == "Closed"].copy()
    
    # Convert dates
    closed_lost[DATE_COL] = pd.to_datetime(closed_lost[DATE_COL], errors="coerce")
    
    # Pivot to create month-by-month view
    pivot = closed_lost.pivot_table(
        index=ROW_DIMS,
        columns=DATE_COL,
        values=VALUE_COL,
        aggfunc="sum",
        fill_value=0
    ).sort_index(axis=1).sort_index()
    
    # Align to our full month grid
    return align_to_months(pivot)

print("✓ get_closed_lost_pipeline() function defined")
print("  Extracts actual closed-lost deals for all months")

✓ get_closed_lost_pipeline() function defined
  Extracts actual closed-lost deals for all months


## 📤 Function: Calculate Outflow (Deals Pushing Out)

**What we're doing:**
Calculate how much pipeline pushes to future months.

**The math:** Pipeline × Push_Rate = Amount leaving (shown as negative)

**Example:** $100k pipeline with 20% push rate → **-$20k** outflow (negative!)

**Important:** 
- ✅ Outflow is ALWAYS negative or zero
- ✅ Any positive values are automatically flipped to negative
- ✅ Current quarter deals don't push (too close to closing)

In [85]:
def calculate_outflow(pipeline, push_rules):
    """
    Calculate how much pipeline pushes to future months.
    Returns NEGATIVE numbers (value leaving the month).
    
    Example: If $100k pushes out → returns -$100k
    
    All months follow their defined push rates - no exceptions.
    """
    # Multiply by push rate and make negative
    outflow = -(pipeline.mul(get_push_rate(pipeline.columns, push_rules), axis=1))
    
    # Force all values to be negative or zero (flip any positive values)
    result = align_to_months(outflow)
    result = result.where(result <= 0, -result.abs())  # Ensure negative
    
    return result

print("✓ calculate_outflow() function defined")
print("  Calculates deals leaving each month (ALWAYS negative)")
print("  All months follow their push rates")

✓ calculate_outflow() function defined
  Calculates deals leaving each month (ALWAYS negative)
  All months follow their push rates


## 📥 Function: Calculate Inflow (Deals Landing)

**What we're doing:**
When deals push out, they land in future months. This calculates where.

**The distribution:**
Pushed deals land at 3, 6, or 9 months later based on distribution rules.

**Example:**
- $100k pushes from October (shown as -$100k outflow)
- **+$60k** lands in January (+3 months) → positive!
- **+$40k** lands in April (+6 months) → positive!

**Important:**
- ✅ Inflow is ALWAYS positive or zero
- ✅ Any negative values are automatically flipped to positive
- ✅ We use abs() on outflow to ensure positive values

In [86]:
def calculate_inflow(outflow, push_rules):
    """
    Take pushed deals and distribute them to future months.
    Returns POSITIVE numbers (value coming into the month).
    
    Deals land at 3, 6, or 9 months later based on distribution.
    
    Example: If $100k pushed from October:
    - $50k lands in January (+3 months) → +$50k
    - $30k lands in April (+6 months) → +$30k
    - $20k lands in July (+9 months) → +$20k
    """
    result = outflow * 0  # Start with zeros
    
    # For each lag period (3, 6, or 9 months)
    for lag in (3, 6, 9):
        # Get the share going to this lag (use abs() to make positive)
        share = outflow.abs().mul(
            get_push_distribution(outflow.columns, push_rules, lag),
            axis=1
        )
        
        # Shift columns forward by lag amount
        # (October's pushed deals → January when lag=3)
        share.columns = [c + DateOffset(months=lag) for c in share.columns]
        
        # Add to result
        result = result.add(share, fill_value=0)
    
    # Force all values to be positive or zero (flip any negative values)
    final = align_to_months(result)
    final = final.where(final >= 0, final.abs())  # Ensure positive
    
    return final

print("✓ calculate_inflow() function defined")
print("  Oct pushes → Jan/Apr/Jul receives (ALWAYS positive)")

✓ calculate_inflow() function defined
  Oct pushes → Jan/Apr/Jul receives (ALWAYS positive)


## 🎯 Function: Calculate Adjusted Pipeline

**What we're doing:**
Combine all push/pull movements to get final adjusted pipeline.

**The complete flow:**
1. Start with existing pipeline
2. Subtract Pre-Q outflow (deals pushing out, NEGATIVE)
3. Add Pre-Q inflow (deals landing back in, POSITIVE)
4. Subtract In-Q outflow (second round of pushes, NEGATIVE)
5. Add In-Q inflow (second round of landings, POSITIVE)

**Result:** The final pipeline amount we'll apply win rates to.

In [87]:
def calculate_adjusted_pipeline(existing, pre_q_out, pre_q_in, in_q_out, in_q_in):
    """
    Calculate final adjusted pipeline after all push/pull movements.
    
    This is like a bank account:
    Starting + Outflows (negative) + Inflows (positive) = Ending balance
    """
    return align_to_months(existing + pre_q_out + pre_q_in + in_q_out + in_q_in)

print("✓ calculate_adjusted_pipeline() function defined")
print("  Combines all flows into final pipeline number")

✓ calculate_adjusted_pipeline() function defined
  Combines all flows into final pipeline number


## 📝 Functions: Format Output for Excel

**What we're doing:**
Make our Excel output clean and readable.

**Two functions:**
1. `format_columns()` - Convert dates to readable format (Jan-2025)
2. `write_to_excel()` - Write dataframe to Excel with nice formatting

In [88]:
def format_columns(df):
    """
    Convert Timestamp columns to readable month-year format.
    Timestamp('2025-10-01') → 'Oct-2025'
    """
    return df.rename(columns=lambda c: 
        c.strftime("%b-%Y") if isinstance(c, (pd.Timestamp, pd.Period)) else c
    )


def write_to_excel(excel_writer, df, sheet_name):
    """
    Write dataframe to Excel with clean formatting.
    - Resets multi-index for readability
    - Formats dates nicely
    - 2 decimal places for numbers
    """
    output = df.reset_index() if df.index.nlevels > 1 else df
    
    format_columns(output).to_excel(
        excel_writer,
        sheet_name=sheet_name,
        index=False,
        merge_cells=False,
        float_format="%.2f"
    )
    
    logging.info(f"  ✓ Wrote '{sheet_name}' ({len(output):,} rows)")

print("✓ Output formatting functions defined")

✓ Output formatting functions defined


## 🚀 MAIN PROCESS: Load Data

**What we're doing:**
1. Load raw data from Excel
2. Clean column names
3. Show summary statistics

In [89]:
logging.info("=" * 70)
logging.info("🚀 STARTING PIPELINE FORECAST")
logging.info("=" * 70)

# Load and clean data
logging.info(f"\n📂 Loading: {INPUT_FILE}")
df_raw = pd.read_excel(INPUT_FILE)
data = clean_headers(df_raw)

# Summary
logging.info(f"\n✓ Loaded {len(data):,} rows")
logging.info(f"✓ Columns: {', '.join(data.columns[:5])}...")

logging.info(f"\n✓ Stage breakdown:")
for stage, count in data[STAGE_COL].value_counts().items():
    logging.info(f"  - {stage}: {count:,} rows")

# Date range in data
dates = pd.to_datetime(data[DATE_COL], errors="coerce").dropna()
if len(dates) > 0:
    logging.info(f"\n✓ Close dates in data:")
    logging.info(f"  - Earliest: {dates.min().strftime('%b %Y')}")
    logging.info(f"  - Latest: {dates.max().strftime('%b %Y')}")
    
    # # Show what gets dropped
    # past_dates = dates[dates < current_month_start]
    # if len(past_dates) > 0:
    #     logging.info(f"\n  ⚠️ {len(past_dates):,} rows have dates before {current_month_start.strftime('%b %Y')}")
    #     logging.info(f"     These will be dropped (we only forecast forward)")

2026-02-26 10:50:58,880 | INFO    | ======================================================================
2026-02-26 10:50:58,884 | INFO    | 🚀 STARTING PIPELINE FORECAST
2026-02-26 10:50:58,886 | INFO    | ======================================================================
2026-02-26 10:50:58,889 | INFO    | 
📂 Loading: C:\Users\SamRobinson\TRICENTIS\Strategic Analytics - Strategic Analytics\Pipeline Model\2026 Planning\November Cleaned\output files\Data Ingestion 2025 Weighted.xlsx
2026-02-26 10:51:03,436 | INFO    | 
✓ Loaded 34,891 rows
2026-02-26 10:51:03,438 | INFO    | ✓ Columns: Geo, Territory, Account_Owner_Bookings_Team__c, Product, Source...
2026-02-26 10:51:03,438 | INFO    | 
✓ Stage breakdown:
2026-02-26 10:51:03,440 | INFO    |   - Closed: 22,387 rows
2026-02-26 10:51:03,441 | INFO    |   - Closed Won: 7,019 rows
2026-02-26 10:51:03,442 | INFO    |   - Open: 5,485 rows
2026-02-26 10:51:03,460 | INFO    | 
✓ Close dates in data:
2026-02-26 10:51:03,462 | INFO    |   -

## 🔄 MAIN PROCESS: Calculate Pipeline Flows

**What we're doing:**
Running all pipeline calculations:
1. Extract existing open pipeline (current month forward only)
2. Calculate pre-quarter outflow (NEGATIVE) & inflow (POSITIVE)
3. Calculate in-quarter outflow (NEGATIVE) & inflow (POSITIVE)
4. Calculate final adjusted pipeline (THIS IS YOUR FORECAST!)

**This is the core of the model!**

In [90]:
logging.info("\n" + "=" * 70)
logging.info("📊 CALCULATING PIPELINE FLOWS")
logging.info("=" * 70)

# Step 1: Get existing pipeline
logging.info("\n[1/7] Extracting existing open pipeline...")
existing = get_existing_pipeline(data)
existing_total = existing.values.sum()
logging.info(f"      → ${existing_total:,.0f} in open pipeline")
logging.info(f"      → {len(existing):,} dimension combinations")

# Step 2: Pre-quarter outflow
logging.info("\n[2/7] Calculating pre-quarter outflow...")
pre_q_out = calculate_outflow(existing, PRE_Q)
pre_q_out_total = abs(pre_q_out.values.sum())
logging.info(f"      → ${pre_q_out_total:,.0f} pushing out (NEGATIVE values)")
if existing_total > 0:
    logging.info(f"      → {pre_q_out_total/existing_total*100:.1f}% of pipeline")

# Step 3: Pre-quarter inflow
logging.info("\n[3/7] Calculating pre-quarter inflow...")
pre_q_in = calculate_inflow(pre_q_out, PRE_Q)
pre_q_in_total = pre_q_in.values.sum()
logging.info(f"      → ${pre_q_in_total:,.0f} flowing back in (POSITIVE values)")

# Step 4: In-quarter outflow
logging.info("\n[4/7] Calculating in-quarter outflow...")
in_q_out = calculate_outflow(existing + pre_q_out + pre_q_in, IN_Q)
in_q_out_total = abs(in_q_out.values.sum())
logging.info(f"      → ${in_q_out_total:,.0f} pushing out (NEGATIVE values, round 2)")

# Step 5: In-quarter inflow
logging.info("\n[5/7] Calculating in-quarter inflow...")
in_q_in = calculate_inflow(in_q_out, IN_Q)
in_q_in_total = in_q_in.values.sum()
logging.info(f"      → ${in_q_in_total:,.0f} flowing back in (POSITIVE values, round 2)")

# Step 6: Final adjusted pipeline (FORECAST)
logging.info("\n[6/7] Calculating final adjusted pipeline...")
adjusted = calculate_adjusted_pipeline(existing, pre_q_out, pre_q_in, in_q_out, in_q_in)
adjusted_total = adjusted.values.sum()
logging.info(f"      → ${adjusted_total:,.0f} FINAL FORECAST")
logging.info(f"      → Net change: ${adjusted_total - existing_total:+,.0f}")

# Step 7: Get closed-won actuals
logging.info("\n[7/7] Extracting closed-won actuals...")
closed_won = get_closed_won_pipeline(data)
closed_won_total = closed_won.values.sum()
logging.info(f"      → ${closed_won_total:,.0f} in closed-won deals")

# Step 8: Get closed-lost actuals  # NEW
logging.info("\n[8/8] Extracting closed-lost actuals...")
closed_lost = get_closed_lost_pipeline(data)          # NEW
closed_lost_total = closed_lost.values.sum()          # NEW
logging.info(f"      → ${closed_lost_total:,.0f} in closed-lost deals")  # NEW

# Summary table
logging.info("\n📈 FLOW SUMMARY:")
logging.info(f"  Starting pipeline:      ${existing_total:>15,.0f}")
logging.info(f"  Pre-Q outflow:          ${-pre_q_out_total:>15,.0f}  ⬅️ NEGATIVE")
logging.info(f"  Pre-Q inflow:           ${pre_q_in_total:>15,.0f}  ➡️ POSITIVE")
logging.info(f"  In-Q outflow:           ${-in_q_out_total:>15,.0f}  ⬅️ NEGATIVE")
logging.info(f"  In-Q inflow:            ${in_q_in_total:>15,.0f}  ➡️ POSITIVE")
logging.info(f"  {'─' * 42}")
logging.info(f"  ADJUSTED FORECAST:      ${adjusted_total:>15,.0f}")
logging.info(f"  Closed-Won Actuals:     ${closed_won_total:>15,.0f}")
logging.info(f"  Closed-Lost Actuals:    ${closed_lost_total:>15,.0f}")   # NEW
logging.info(f"\n  ✅ Sign checks passed: Outflows negative, Inflows positive")

2026-02-26 10:51:03,479 | INFO    | 
2026-02-26 10:51:03,480 | INFO    | 📊 CALCULATING PIPELINE FLOWS
2026-02-26 10:51:03,481 | INFO    | ======================================================================
2026-02-26 10:51:03,483 | INFO    | 
[1/7] Extracting existing open pipeline...
2026-02-26 10:51:03,500 | INFO    |       → $268,855,973 in open pipeline
2026-02-26 10:51:03,502 | INFO    |       → 1,640 dimension combinations
2026-02-26 10:51:03,503 | INFO    | 
[2/7] Calculating pre-quarter outflow...
2026-02-26 10:51:03,507 | INFO    |       → $85,891,200 pushing out (NEGATIVE values)
2026-02-26 10:51:03,507 | INFO    |       → 31.9% of pipeline
2026-02-26 10:51:03,508 | INFO    | 
[3/7] Calculating pre-quarter inflow...
2026-02-26 10:51:03,515 | INFO    |       → $67,029,281 flowing back in (POSITIVE values)
2026-02-26 10:51:03,516 | INFO    | 
[4/7] Calculating in-quarter outflow...
2026-02-26 10:51:03,520 | INFO    |       → $135,850,747 pushing out (NEGATIVE values, round 2

## 💾 FINAL STEP: Write to Excel

**What we're doing:**
Writing all calculations to Excel with these sheets:

**Pipeline flows:**
- existing_pipeline
- pre_q_outflow (NEGATIVE), pre_q_inflow (POSITIVE)
- in_q_outflow (NEGATIVE), in_q_inflow (POSITIVE)
- adjusted_pipeline (FINAL FORECAST!)

**Note:** All sheets only contain current month forward. Past months dropped.

In [91]:
logging.info("\n" + "=" * 70)
logging.info("💾 WRITING OUTPUT TO EXCEL")
logging.info("=" * 70)

output_path = Path(OUTPUT_FILE)
logging.info(f"\nFile: {output_path}")

with pd.ExcelWriter(output_path, engine="xlsxwriter", mode="w") as xl:
    
    # Intermediate pipeline flows
    logging.info("\n📊 Writing pipeline flow sheets:")
    write_to_excel(xl, existing, "existing_pipeline")
    write_to_excel(xl, pre_q_out, "pre_q_outflow")
    write_to_excel(xl, pre_q_in, "pre_q_inflow")
    write_to_excel(xl, in_q_out, "in_q_outflow")
    write_to_excel(xl, in_q_in, "in_q_inflow")
    write_to_excel(xl, adjusted, "adjusted_pipeline")
    
    # Actuals
    logging.info("\n🎯 Writing actuals:")
    write_to_excel(xl, closed_won, "closed_won_pipe")
    write_to_excel(xl, closed_lost, "closed_lost_pipe")   # ← NEW

logging.info("\n" + "=" * 70)
logging.info("✅ FORECAST COMPLETE!")
logging.info("=" * 70)
logging.info(f"\n📁 Output: {OUTPUT_FILE}")
logging.info(f"📊 Total rows in adjusted forecast: {len(adjusted):,}")
logging.info(f"📊 Total rows in closed-won: {len(closed_won):,}")
logging.info(f"📊 Total rows in closed-lost: {len(closed_lost):,}")   # NEW
logging.info(f"💰 Closed-lost total: ${closed_lost_total:,.0f}")     # NEW
logging.info(f"📅 Months covered: {len(month_grid)}")
logging.info(f"💰 Adjusted pipeline total: ${adjusted_total:,.0f}")
logging.info(f"💰 Closed-won total: ${closed_won_total:,.0f}")

logging.info(f"\n📌 Key sheets:")
logging.info(f"  • 'adjusted_pipeline' = Final forecast after all push/pull")
logging.info(f"  • 'closed_won_pipe' = Actual closed-won deals")
logging.info(f"  • 'closed_lost_pipe' = Actual closed-lost deals")     # NEW
logging.info(f"  • 'existing_pipeline', 'pre_q_*', 'in_q_*' = Intermediate calculations")
logging.info(f"\n🎉 Done! Open Excel to view your complete forecast.")

2026-02-26 10:51:03,609 | INFO    | 
2026-02-26 10:51:03,611 | INFO    | 💾 WRITING OUTPUT TO EXCEL
2026-02-26 10:51:03,612 | INFO    | ======================================================================
2026-02-26 10:51:03,613 | INFO    | 
File: C:\Users\SamRobinson\TRICENTIS\Strategic Analytics - Strategic Analytics\Pipeline Model\2026 Planning\November Cleaned\Open Pipeline Forecast Product V10.xlsx
2026-02-26 10:51:03,616 | INFO    | 
📊 Writing pipeline flow sheets:
2026-02-26 10:51:03,759 | INFO    |   ✓ Wrote 'existing_pipeline' (1,640 rows)
2026-02-26 10:51:03,900 | INFO    |   ✓ Wrote 'pre_q_outflow' (1,640 rows)
2026-02-26 10:51:04,034 | INFO    |   ✓ Wrote 'pre_q_inflow' (1,640 rows)
2026-02-26 10:51:04,186 | INFO    |   ✓ Wrote 'in_q_outflow' (1,640 rows)
2026-02-26 10:51:04,322 | INFO    |   ✓ Wrote 'in_q_inflow' (1,640 rows)
2026-02-26 10:51:04,452 | INFO    |   ✓ Wrote 'adjusted_pipeline' (1,640 rows)
2026-02-26 10:51:04,453 | INFO    | 
🎯 Writing actuals:
2026-02-26 10

## Target Explosion

In [92]:
# ===============================================================
# TARGETS → APPLY SPLITS (wrapper)
#   • Def: Territory-based targets → split by Tier → Product → Source → Deal Type
#   • Reads:
#       - targets_xlsx: [Workbook]\Input files\Targets.xlsx (territory totals by month)
#       - ingest_xlsx : [Workbook]\output files\Data Ingestion.xlsx (history to learn splits)
#   • Writes (optional, preserves others):
#       - "Targets"            (cleaned territory totals; wide)
#       - "Targets_Split_Long" (long: KEYS + Month_TS + Target)
#       - "Targets_Split"      (wide: KEYS + Mon-YYYY)
#   • Month headers standardized to "Mon-YYYY"
#   • Flow KEYS: ["Territory","Tier","Product","Source","Deal Type"]
# ===============================================================

from pathlib import Path
import pandas as pd
from openpyxl import load_workbook, Workbook

# ---------- Defaults (override per call) ----------
from pathlib import Path

def find_root(markers=("Input files", "output files", "Python Scripts", ".git", ".project-root")) -> Path:
    p = Path.cwd().resolve()
    for q in (p, *p.parents):
        if any((q/m).exists() for m in markers):   # any marker present
            return q
    raise FileNotFoundError(f"No root found from {p}")

ROOT = find_root()  # add a zero-byte file named ".project-root" at your root (recommended)
DEFAULT_OUT_XL = ROOT / "Open Pipeline Forecast Product V10.xlsx"
DEFAULT_TARGETS  = DEFAULT_OUT_XL.parent / "Input files" / "Targets Product V2.xlsx"
DEFAULT_INGEST   = DEFAULT_OUT_XL.parent / "output files" / "Data Ingestion 2025 Weighted.xlsx"

ID_COLS = ["Territory", "Tier", "Product", "Source", "Deal Type"]

# ---------- helpers ----------
def _month_to_ts(col):
    try:
        dt = pd.to_datetime(col, errors="coerce")
        if pd.notna(dt): return pd.Timestamp(dt.year, dt.month, 1)
    except Exception: pass
    for fmt in ("%b-%Y", "%Y-%m", "%m/%d/%Y", "%Y/%m/%d"):
        try:
            dt = pd.to_datetime(col, format=fmt); return pd.Timestamp(dt.year, dt.month, 1)
        except Exception: continue
    return None

def _fmt_month_headers(df):
    return df.rename(columns=lambda c: c.strftime("%b-%Y") if isinstance(c, pd.Timestamp) else c)

def _ensure_cols(df, cols):
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            out[c] = f"Unspecified_{c}"
    return out

def _ensure_split_defaults(df_hist, by_keys, value_col, out_share_name, fill_label):
    """
    Build shares for a split step; if a parent combo has no history, backfill with 1.0.
    by_keys order: parents... , child (we split into last key).
    """
    if df_hist.empty:
        s = {k: [fill_label] for k in by_keys}
        s[out_share_name] = [1.0]
        return pd.DataFrame(s)

    grp = df_hist.groupby(by_keys, dropna=False, as_index=False)[value_col].sum()
    parents = by_keys[:-1]
    share = grp.assign(**{
        out_share_name: lambda d: d[value_col] / d.groupby(parents, dropna=False)[value_col].transform("sum")
    }).drop(columns=value_col)

    # backfill missing parents with a single 1.0 row
    parents_only = grp[parents].drop_duplicates()
    present = share[parents].drop_duplicates() if not share.empty else parents_only.iloc[0:0]
    missing = parents_only.merge(present, on=parents, how="left", indicator=True)
    missing = missing[missing["_merge"] == "left_only"].drop(columns="_merge")
    if not missing.empty:
        missing = missing.copy()
        missing[by_keys[-1]] = fill_label
        missing[out_share_name] = 1.0
        share = pd.concat([share, missing], ignore_index=True)

    if share.empty:
        s = {k: [fill_label] for k in by_keys}
        s[out_share_name] = [1.0]
        return pd.DataFrame(s)

    return share

def _write_frames_xlsx_preserving(path: Path, frames: dict[str, pd.DataFrame]):
    if path.exists():
        with pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace") as xl:
            for s, df_out in frames.items():
                df_out.to_excel(xl, sheet_name=s, index=False)
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        with pd.ExcelWriter(path, engine="openpyxl", mode="w") as xl:
            for s, df_out in frames.items():
                df_out.to_excel(xl, sheet_name=s, index=False)

def _write_frames_xlsm_preserving(path: Path, frames: dict[str, pd.DataFrame]):
    if path.exists():
        wb = load_workbook(path, keep_vba=True)
    else:
        wb = Workbook()
        if "Sheet" in wb.sheetnames: del wb["Sheet"]
    for s, df_out in frames.items():
        if s in wb.sheetnames: del wb[s]
        ws = wb.create_sheet(s); ws.append(list(df_out.columns))
        for row in df_out.itertuples(index=False, name=None): ws.append(list(row))
    wb.save(path)

def _write_frames_preserving(path: Path, frames: dict[str, pd.DataFrame]):
    suffix = path.suffix.lower()
    if suffix == ".xlsm":
        _write_frames_xlsm_preserving(path, frames)
    else:
        _write_frames_xlsx_preserving(path, frames)
    # verify
    wb2 = load_workbook(path, read_only=True, keep_vba=(suffix==".xlsm"))
    try: sheetnames = wb2.sheetnames
    finally: wb2.close()
    for s in frames:
        if s not in sheetnames:
            raise RuntimeError(f"Write failed: '{s}' not found after save in {path}")

# ---------- public API ----------
def build_targets_splits_df(
    targets_xlsx: Path = DEFAULT_TARGETS,
    ingest_xlsx: Path = DEFAULT_INGEST,
    out_keys: list[str] = ID_COLS,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build:
      - targets_wide        (Territory + Product + placeholders + months)
      - targets_split_long  (Territory,Tier,Product,Source,Deal Type, Month_TS, Target)
      - targets_split_wide  (KEYS + Mon-YYYY)
    No writes performed here.

    Assumes Targets Product file has:
      - 'Territory'
      - 'Product'
      - month columns (datetime-like or parseable to month)
    """
    targets_xlsx = Path(targets_xlsx)
    ingest_xlsx  = Path(ingest_xlsx)

    # 1) Load Targets Product.xlsx (first sheet or "Targets")
    if not targets_xlsx.exists():
        raise FileNotFoundError(f"Targets.xlsx not found at:\n{targets_xlsx}")
    xl_t = pd.ExcelFile(targets_xlsx)
    t_sheet = "Targets" if "Targets" in xl_t.sheet_names else xl_t.sheet_names[0]
    t_raw   = xl_t.parse(t_sheet)

    if "Territory" not in t_raw.columns:
        raise KeyError("Targets.xlsx must include a 'Territory' column.")
    if "Product" not in t_raw.columns:
        raise KeyError("Targets.xlsx must include a 'Product' column for product-level targets.")

    # Identify/standardize month columns → Timestamps
    non_month_cols = {"Territory", "Product"}
    month_map = {c: _month_to_ts(c) for c in t_raw.columns if c not in non_month_cols}
    keep_ts   = [ts for ts in month_map.values() if ts is not None]
    if not keep_ts:
        raise ValueError("No parseable month columns found in Targets.xlsx.")

    t = t_raw.rename(columns={c: month_map[c] for c in month_map if month_map[c] is not None}).copy()
    month_cols = sorted([c for c in t.columns if isinstance(c, pd.Timestamp)])

    # Pad remaining ID cols (Tier, Source, Deal Type) with placeholders; keep only IDs + months
    # ID_COLS = ["Territory", "Tier", "Product", "Source", "Deal Type"]
    t = _ensure_cols(t, [c for c in out_keys if c not in ("Territory", "Product")])
    t = t[["Territory", "Product"] + [c for c in out_keys if c not in ("Territory", "Product")] + month_cols]

    # Clean territory+product-wide copy (wide) for transparency
    targets_wide = _fmt_month_headers(
        t[["Territory", "Product"] + [c for c in out_keys if c not in ("Territory", "Product")] + month_cols]
    )

    # 2) Build SPLITS from Data Ingestion.xlsx (for Tier / Source / Deal Type)
    if not ingest_xlsx.exists():
        raise FileNotFoundError(f"Data Ingestion.xlsx not found at:\n{ingest_xlsx}")
    df_ing = pd.read_excel(ingest_xlsx)

    # Normalize key names we need
    df_ing = df_ing.rename(columns=lambda c: "Deal Type" if str(c).strip() == "DealType" else c)
    if "Create_Month" in df_ing.columns:
        df_ing["Create_Month"] = pd.to_datetime(df_ing["Create_Month"], errors="coerce")
    elif "Opp Close Date" in df_ing.columns:
        df_ing["Create_Month"] = pd.to_datetime(df_ing["Opp Close Date"], errors="coerce")
    else:
        raise KeyError("Data Ingestion must have 'Create_Month' or 'Opp Close Date'.")
    if "Product NACV" not in df_ing.columns:
        raise KeyError("Data Ingestion must include 'Product NACV' for split weighting.")
    df_ing = _ensure_cols(df_ing, out_keys)

    # Use last calendar year for splits
    last_year = int(df_ing["Create_Month"].dt.year.max()) - 1
    hist = df_ing[df_ing["Create_Month"].dt.year.eq(last_year)].copy()

    # Tier within (Territory, Product)
    tier_split = _ensure_split_defaults(
        hist[["Territory", "Product", "Tier", "Product NACV"]].rename(columns={"Product NACV": "x"}),
        by_keys=["Territory", "Product", "Tier"],
        value_col="x",
        out_share_name="TierShare",
        fill_label="Unspecified_Tier"
    )

    # Source within (Territory, Product, Tier)
    src_split = _ensure_split_defaults(
        hist[["Territory", "Product", "Tier", "Source", "Product NACV"]].rename(columns={"Product NACV": "x"}),
        by_keys=["Territory", "Product", "Tier", "Source"],
        value_col="x",
        out_share_name="SourceShare",
        fill_label="Unspecified_Source"
    )

    # Deal Type within (Territory, Product, Tier)
    deal_split = _ensure_split_defaults(
        hist[["Territory", "Product", "Tier", "Deal Type", "Product NACV"]].rename(columns={"Product NACV": "x"}),
        by_keys=["Territory", "Product", "Tier", "Deal Type"],
        value_col="x",
        out_share_name="DealShare",
        fill_label="Unspecified_Deal Type"
    )

    # 3) Apply cascade splits starting from (Territory, Product, Month_TS)
    long_base = (
        t[["Territory", "Product"] + month_cols]
          .melt(
              id_vars=["Territory", "Product"],
              value_vars=month_cols,
              var_name="Month_TS",
              value_name="Target_Total"
          )
    )

    alloc = (
        long_base
        # expand to tiers
        .merge(tier_split, on=["Territory", "Product"], how="left")
        # expand to sources
        .merge(src_split, on=["Territory", "Product", "Tier"], how="left")
        # expand to deal types
        .merge(deal_split, on=["Territory", "Product", "Tier"], how="left")
        .fillna({"TierShare": 1.0, "SourceShare": 1.0, "DealShare": 1.0})
    )

    alloc["Allocated_Target"] = (
        alloc["Target_Total"] * alloc["TierShare"] * alloc["SourceShare"] * alloc["DealShare"]
    )

    targets_split_long = (
        alloc[["Territory", "Tier", "Product", "Source", "Deal Type", "Month_TS", "Allocated_Target"]]
          .rename(columns={"Allocated_Target": "Target"})
          .sort_values(
              ["Territory", "Tier", "Product", "Source", "Deal Type", "Month_TS"],
              kind="mergesort"
          )
    )

    targets_split_long["Month_TS"] = pd.to_datetime(targets_split_long["Month_TS"], errors="coerce")

    targets_split_wide = (
        targets_split_long
          .assign(Month_Str=lambda d: d["Month_TS"].dt.strftime("%b-%Y"))
          .drop(columns="Month_TS")
          .pivot_table(
              index=ID_COLS,
              columns="Month_Str",
              values="Target",
              aggfunc="sum",
              fill_value=0
          )
          .reset_index()
    )

    # order month columns ascending
    month_str_cols = [
        c for c in targets_split_wide.columns
        if isinstance(c, str) and len(c) == 8 and c[3] == "-"
    ]
    month_str_sorted = sorted(month_str_cols, key=lambda s: pd.to_datetime(s, format="%b-%Y"))
    targets_split_wide = targets_split_wide[ID_COLS + month_str_sorted]

    # Pretty-format the wide table’s month headers
    targets_wide = _fmt_month_headers(targets_wide)

    return targets_wide, targets_split_long, targets_split_wide

def write_targets_splits(
    out_xl: Path = DEFAULT_OUT_XL,
    targets_wide: pd.DataFrame | None = None,
    targets_split_long: pd.DataFrame | None = None,
    targets_split_wide: pd.DataFrame | None = None,
):
    """Write ONLY Targets, Targets_Split_Long, Targets_Split (preserve others)."""
    frames_out = {}
    if targets_wide is not None:        frames_out["Targets"]            = targets_wide
    if targets_split_long is not None:  frames_out["Targets_Split_Long"] = targets_split_long
    if targets_split_wide is not None:  frames_out["Targets_Split"]      = targets_split_wide

    suffix = out_xl.suffix.lower()
    if suffix == ".xlsm":
        _write_frames_xlsm_preserving(out_xl, frames_out)
    else:
        _write_frames_xlsx_preserving(out_xl, frames_out)

    # verify
    wb = load_workbook(out_xl, read_only=True, keep_vba=(suffix==".xlsm"))
    try:
        missing = [s for s in frames_out if s not in wb.sheetnames]
        if missing:
            raise RuntimeError(f"Write failed: missing sheets after save: {missing}")
    finally:
        wb.close()
    print("✓ Wrote sheets:", list(frames_out.keys()))

def build_and_write_targets_splits(
    out_xl: Path = DEFAULT_OUT_XL,
    targets_xlsx: Path = DEFAULT_TARGETS,
    ingest_xlsx: Path = DEFAULT_INGEST,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Convenience: build and write targets splits; returns the three DFs."""
    tw, tsl, tsw = build_targets_splits_df(targets_xlsx=targets_xlsx, ingest_xlsx=ingest_xlsx)
    write_targets_splits(out_xl=out_xl, targets_wide=tw, targets_split_long=tsl, targets_split_wide=tsw)
    return tw, tsl, tsw

# Build + write from disk sources
tw, tsl, tsw = build_and_write_targets_splits()

# Or just build (inspect/edit), then write later
tw, tsl, tsw = build_targets_splits_df()
write_targets_splits(targets_wide=tw, targets_split_long=tsl, targets_split_wide=tsw)

✓ Wrote sheets: ['Targets', 'Targets_Split_Long', 'Targets_Split']
✓ Wrote sheets: ['Targets', 'Targets_Split_Long', 'Targets_Split']
